[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/TRELLIS-2_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TRELLIS-2_Colab.ipynb)

# 🧊 Microsoft TRELLIS.2 (4B) — Image-to-PBR-3D & O-Voxel Asset Generation (MIT)

**[TRELLIS.2](https://microsoft.github.io/TRELLIS.2)** (`microsoft/TRELLIS.2-4B`, [arXiv 2512.14692](https://arxiv.org/abs/2512.14692)) is Microsoft Research's flagship **4-billion-parameter** 3D generative foundation model. Built on a novel field-free sparse representation called **O-Voxel** (Omni-Voxel) and a 16× spatial-downsampling Sparse 3D VAE, TRELLIS.2 generates high-resolution 3D meshes up to **$1536^3$** with arbitrary topology (open surfaces, non-manifold geometry, internal enclosed structures) and full **PBR materials** (**Base Color, Roughness, Metallic, and Opacity**).

* **Paper:** [Native and Compact Structured Latents for 3D Generation (arXiv:2512.14692)](https://arxiv.org/abs/2512.14692)
* **Code:** [github.com/microsoft/TRELLIS.2](https://github.com/microsoft/TRELLIS.2) &nbsp;·&nbsp; **Weights:** [🤗 microsoft/TRELLIS.2-4B](https://huggingface.co/microsoft/TRELLIS.2-4B) &nbsp;·&nbsp; **Space:** [🤗 spaces/microsoft/TRELLIS.2](https://huggingface.co/spaces/microsoft/TRELLIS.2)
* **License:** **MIT License** (both code and 4B weights — commercial-friendly!)

---

### ✨ Key Capabilities in This Notebook
1. **3-Stage Image-to-PBR-3D Pipeline (`Trellis2ImageTo3DPipeline`)** — Sparse Structure DiT $\rightarrow$ Shape SLat DiT ($512^3 \rightarrow 1024^3 / 1536^3$ cascade) $\rightarrow$ Material/Texture SLat DiT $\rightarrow$ `o_voxel.postprocess.to_glb(...)` with WebP-compressed PBR textures.
2. **Shape-Conditioned PBR Texturing (`Trellis2TexturingPipeline`)** — Upload any untextured `.glb / .obj / .ply` mesh + a reference image to synthesize UV-baked PBR textures via `texturing_pipeline.json`.
3. **Instant Prebuilt CUDA Wheels** — Installs precompiled `o_voxel`, `cumesh`, `flex_gemm`, and `nvdiffrec_render` wheels from our GitHub Releases (`wheels-t4-v1.1`, `wheels-l4-v1.1`, `wheels-a100-v1.1`) in seconds — **no 25-minute CUDA compilation required**.
4. **Zero-Token Ungated Fallback** — Automatically falls back to ungated community mirrors (`camenduru/dinov3-vitl16-pretrain-lvd1689m` and `ZhengPeng7/BiRefNet`) when `HF_TOKEN` is not set, so the notebook works out of the box without gated model approvals.

---

### 🖥️ GPU & Resolution Recommendations

| GPU | VRAM | Recommended Preset | Low-VRAM Mode | Typical Runtime |
|---|---|---|---|---|
| **A100** | 40 GB | `1536_cascade` or `1024_cascade` | Optional (`False` for max speed) | ~35–65 s |
| **L4** | 24 GB | `1024_cascade` (or `1536_cascade` w/ Low-VRAM) | **Enabled (`True`)** | ~45–90 s |
| **T4** | 15 GB | `512` or `1024_cascade` | **Enabled (`True`, auto)** | ~60–120 s |

In [ ]:
# @title STEP 1 — Install Prebuilt CUDA Wheels & Dependencies
# @markdown Detects your Colab GPU (`T4` / `L4` / `A100`), installs our prebuilt `o_voxel`, `cumesh`, `flex_gemm`, and `nvdiffrec_render` CUDA wheels from GitHub Releases, compiles `nvdiffrast`, and clones [microsoft/TRELLIS.2](https://github.com/microsoft/TRELLIS.2).

import os, sys, subprocess, shutil, pathlib, traceback
import torch
from tqdm.auto import tqdm

print("=" * 72)
print("Microsoft TRELLIS.2 (4B) — STEP 1: Install Dependencies")
print("=" * 72)
print(f"  Python : {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}  (CUDA: {torch.version.cuda})")

if not torch.cuda.is_available():
    raise SystemExit("[ERROR] No CUDA GPU detected. Change Colab runtime to GPU (T4, L4, or A100).")

# Mount Google Drive for persistent HuggingFace weight caching
CONNECT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
if CONNECT_GOOGLE_DRIVE and not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as _e:
        print(f"[Drive] Mount skipped or running locally: {_e}")

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/TRELLIS2') if os.path.exists('/content/drive/MyDrive') else pathlib.Path('/content/AEI_3D_Cache/TRELLIS2')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.makedirs(os.environ['HUGGINGFACE_HUB_CACHE'], exist_ok=True)
os.makedirs('/content/images_in', exist_ok=True)
print(f"[Cache] HF_HOME -> {os.environ['HF_HOME']}")

# Detect GPU architecture and map to prebuilt wheel release tag
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f"{cap[0]}{cap[1]}"
arch_tag = {'75': 't4', '80': 'a100', '86': 'a100', '89': 'l4'}.get(arch, None)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[GPU] Detected {gpu_name} ({vram_gb:.1f} GB VRAM) -> sm_{arch} (wheel tag: {arch_tag})")

# Pin torch to 2.11.0+cu128 so prebuilt C++ / CUDA wheels match c10::IValue ABI
_target_torch = "2.11.0"
if arch_tag is not None and not torch.__version__.startswith(_target_torch):
    print(f"[Fix] Colab has torch {torch.__version__}; aligning to {_target_torch}+cu128 for prebuilt CUDA wheels...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "--no-input",
        f"torch=={_target_torch}+cu128",
        "torchvision==0.26.0+cu128",
        "torchaudio==2.11.0+cu128",
        "--index-url", "https://download.pytorch.org/whl/cu128",
        "--force-reinstall",
    ], check=False)
    print("[Fix] Restarting kernel via os.execv so torch 2.11.0+cu128 takes effect...")
    os.execv(sys.executable, [sys.executable] + sys.argv)

# Keep Colab resolver quiet on pre-installed packages
_pip_quiet = ["-q", "--disable-pip-version-check", "--no-input"]
subprocess.run([sys.executable, "-m", "pip", "install"] + _pip_quiet + [
    "pydantic>=2.12,<3", "starlette>=1.0.1,<2", "trimesh", "pygltflib", "plyfile",
    "easydict", "huggingface_hub", "tqdm==4.66.5", "ninja"
], check=False)

# Install prebuilt CUDA wheels (o_voxel, cumesh, flex_gemm, nvdiffrec_render)
WHEELS = {
    'o_voxel': 'o_voxel-0.0.1-cp312-cp312-linux_x86_64.whl',
    'cumesh': 'cumesh-0.0.1-cp312-cp312-linux_x86_64.whl',
    'flex_gemm': 'flex_gemm-1.0.0-cp312-cp312-linux_x86_64.whl',
    'nvdiffrec_render': 'nvdiffrec_render-0.0.0-cp312-cp312-linux_x86_64.whl',
}

if arch_tag is not None:
    for name, whl in tqdm(WHEELS.items(), desc="Installing CUDA wheels"):
        installed = False
        for ver in ('v1.1', 'v1.0'):
            url = f"https://github.com/Skquark/AEI-Colab-Notebooks/releases/download/wheels-{arch_tag}-{ver}/{whl}"
            try:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", url], check=True)
                print(f"  [OK] {name} installed from wheels-{arch_tag}-{ver}")
                installed = True
                break
            except Exception as e:
                last_err = e
        if not installed:
            print(f"  [WARN] Could not install prebuilt {name}: {last_err}")
else:
    print(f"[WARN] No prebuilt wheels for sm_{arch}; use Pixal3D_Wheel_Builder.ipynb to compile for this GPU.")

# Compile nvdiffrast from NVlabs source against the active torch
print("[nvdiffrast] Installing from NVlabs/nvdiffrast...")
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{cap[0]}.{cap[1]}"
try:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/NVlabs/nvdiffrast.git"
    ], check=True)
    print("[nvdiffrast] Installed successfully.")
except Exception as e:
    print(f"[WARN] nvdiffrast build warning: {e}")

# Pin transformers==4.57.3 so DINOv3ViTModel exposes self.model.layer directly
import transformers as _tf_check
if not _tf_check.__version__.startswith("4.57"):
    print(f"[Fix] Pinning transformers from {_tf_check.__version__} to 4.57.3 for DINOv3ViTModel compatibility...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.3"], check=True)
    for m in list(sys.modules):
        if m.startswith("transformers"):
            del sys.modules[m]

# Install TRELLIS.2 Python dependencies (with pinned gradio==5.49.1)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "gradio==5.49.1", "trimesh", "pygltflib", "plyfile", "moderngl",
    "huggingface_hub", "kornia", "kornia-rs", "imageio", "imageio-ffmpeg",
    "tqdm==4.66.5", "easydict", "opencv-python-headless", "zstandard",
    "timm", "diffusers", "accelerate", "einops", "onnxruntime", "rembg",
    "scipy", "scikit-image", "matplotlib"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
    "https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl"
], check=False)

# Clone or update microsoft/TRELLIS.2 repository
REPO_DIR = '/content/TRELLIS.2'
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print("[Git] Cloning https://github.com/microsoft/TRELLIS.2.git ...")
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/microsoft/TRELLIS.2.git', REPO_DIR], check=True)
else:
    print("[Git] Updating existing /content/TRELLIS.2 clone (git pull --ff-only)...")
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify CUDA extension imports
print("\n[Verify] Checking CUDA extensions:")
for mod_name in ('o_voxel', 'cumesh', 'flex_gemm', 'nvdiffrec_render', 'nvdiffrast.torch'):
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name}")
    except Exception as _e:
        print(f"  ✗ {mod_name}: {_e}")

print(f"\n[DONE] STEP 1 complete on {gpu_name} ({vram_gb:.1f} GB VRAM) — proceed to STEP 2.")

In [ ]:
# @title STEP 2 — Pre-Download & Cache TRELLIS.2-4B Weights to Google Drive
# @markdown Downloads `microsoft/TRELLIS.2-4B` checkpoints, the sparse-structure decoder from `microsoft/TRELLIS-image-large`, and the DINOv3 + BiRefNet conditioning models into your Google Drive cache (`/content/drive/MyDrive/AEI_3D_Cache/TRELLIS2`). Subsequent Colab sessions reuse the cached files immediately.

import os, sys, json, pathlib, subprocess
from huggingface_hub import snapshot_download, hf_hub_download
from tqdm.auto import tqdm

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/TRELLIS2') if os.path.exists('/content/drive/MyDrive') else pathlib.Path('/content/AEI_3D_Cache/TRELLIS2')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
os.makedirs(os.environ['HUGGINGFACE_HUB_CACHE'], exist_ok=True)

REPO_DIR = '/content/TRELLIS.2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

PRECACHE_FULL_4B_WEIGHTS = True  # @param {type:"boolean"}
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()

print("=" * 72)
print("STEP 2 — Caching TRELLIS.2-4B Weights & Conditioning Models")
print("=" * 72)
print(f"  HF_HOME : {os.environ['HF_HOME']}")

# 1. Pull git-lfs assets (HDRI maps & example images) in /content/TRELLIS.2
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'lfs', 'pull'], check=False, capture_output=True)

# 2. Download pipeline configs from microsoft/TRELLIS.2-4B
for cfg_name in ('pipeline.json', 'texturing_pipeline.json'):
    try:
        p = hf_hub_download('microsoft/TRELLIS.2-4B', cfg_name)
        print(f"  [Config] {cfg_name} -> {p}")
    except Exception as e:
        print(f"  [WARN] Could not download {cfg_name}: {e}")

# 3. Download TRELLIS-image-large sparse structure decoder (referenced in pipeline.json)
print("\n[1/4] Caching sparse_structure_decoder from microsoft/TRELLIS-image-large ...")
snapshot_download(
    repo_id='microsoft/TRELLIS-image-large',
    allow_patterns=['ckpts/ss_dec_conv3d_16l8_fp16*'],
    tqdm_class=tqdm,
    max_workers=4,
)

# 4. Download microsoft/TRELLIS.2-4B checkpoints
if PRECACHE_FULL_4B_WEIGHTS:
    print("\n[2/4] Caching microsoft/TRELLIS.2-4B checkpoints (~8 GB, cached on Drive) ...")
    snapshot_download(
        repo_id='microsoft/TRELLIS.2-4B',
        allow_patterns=['*.json', 'ckpts/*'],
        tqdm_class=tqdm,
        max_workers=4,
    )
else:
    print("\n[2/4] Skipping full 4B pre-download (weights will download on-demand in Step 3).")

# 5. Cache DINOv3 image feature extractor (try official gated repo if HF_TOKEN is set, else ungated mirror)
print("\n[3/4] Caching DINOv3 ViT-L/16 feature extractor ...")
DINOV3_MODEL_ID = 'camenduru/dinov3-vitl16-pretrain-lvd1689m'
if os.environ.get('HF_TOKEN'):
    try:
        snapshot_download(
            'facebook/dinov3-vitl16-pretrain-lvd1689m',
            allow_patterns=['*.json', '*.safetensors', '*.bin', '*.txt'],
            tqdm_class=tqdm,
            max_workers=4,
        )
        DINOV3_MODEL_ID = 'facebook/dinov3-vitl16-pretrain-lvd1689m'
        print(f"  [OK] Cached official {DINOV3_MODEL_ID}")
    except Exception as _e:
        print(f"  [Info] facebook/dinov3 is gated ({_e}); using ungated mirror {DINOV3_MODEL_ID}")
if DINOV3_MODEL_ID == 'camenduru/dinov3-vitl16-pretrain-lvd1689m':
    snapshot_download(
        DINOV3_MODEL_ID,
        allow_patterns=['*.json', '*.safetensors', '*.bin', '*.txt', '*.model'],
        tqdm_class=tqdm,
        max_workers=4,
    )
    print(f"  [OK] Cached ungated DINOv3 mirror: {DINOV3_MODEL_ID}")

# 6. Cache BiRefNet background removal model (try briaai/RMBG-2.0 if HF_TOKEN set, else ZhengPeng7/BiRefNet)
print("\n[4/4] Caching BiRefNet background removal model ...")
REMBG_MODEL_ID = 'ZhengPeng7/BiRefNet'
if os.environ.get('HF_TOKEN'):
    try:
        snapshot_download(
            'briaai/RMBG-2.0',
            allow_patterns=['*.json', '*.safetensors', '*.pth', '*.bin', '*.py'],
            tqdm_class=tqdm,
            max_workers=2,
        )
        REMBG_MODEL_ID = 'briaai/RMBG-2.0'
        print(f"  [OK] Cached {REMBG_MODEL_ID}")
    except Exception as _e:
        print(f"  [Info] briaai/RMBG-2.0 is gated ({_e}); using ungated {REMBG_MODEL_ID}")
if REMBG_MODEL_ID == 'ZhengPeng7/BiRefNet':
    snapshot_download(
        REMBG_MODEL_ID,
        allow_patterns=['*.json', '*.safetensors', '*.pth', '*.bin', '*.py'],
        tqdm_class=tqdm,
        max_workers=2,
    )
    print(f"  [OK] Cached ungated BiRefNet: {REMBG_MODEL_ID}")

# Verify example images & HDRI maps
ex_dir = pathlib.Path(REPO_DIR) / 'assets' / 'example_image'
hdri_dir = pathlib.Path(REPO_DIR) / 'assets' / 'hdri'
n_ex = len(list(ex_dir.glob('*.*'))) if ex_dir.exists() else 0
n_hdri = len(list(hdri_dir.glob('*.exr'))) if hdri_dir.exists() else 0
print(f"\n[Assets] {n_ex} example image(s) in {ex_dir} | {n_hdri} HDRI .exr map(s) in {hdri_dir}")
os.makedirs('/content/trellis2_output', exist_ok=True)
print("[DONE] STEP 2 complete — proceed to STEP 3.")

In [ ]:
# @title STEP 3 — Core TRELLIS.2 Pipeline Engine (Image-to-3D & PBR Texturing)
# @markdown Initializes `Trellis2ImageTo3DPipeline` and `Trellis2TexturingPipeline`, configures attention backends and ungated DINOv3/BiRefNet fallbacks, and defines `run_trellis2_image_to_3d()` and `run_trellis2_texturing()`.

import os, sys, gc, time, math, json, glob, shutil, pathlib, warnings, traceback
from datetime import datetime
import numpy as np
import torch
import cv2
import imageio
import trimesh
from PIL import Image

warnings.filterwarnings('ignore', category=SyntaxWarning)
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/TRELLIS2') if os.path.exists('/content/drive/MyDrive') else pathlib.Path('/content/AEI_3D_Cache/TRELLIS2')
os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(CACHE_ROOT / 'huggingface' / 'hub'))

REPO_DIR = '/content/TRELLIS.2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Auto-detect optimal attention backend
attn_backend = 'sdpa'
try:
    import flash_attn_interface
    attn_backend = 'flash_attn_3'
    print('[Attn] Using FlashAttention 3')
except ImportError:
    try:
        import flash_attn
        attn_backend = 'flash_attn'
        print('[Attn] Using FlashAttention 2')
    except ImportError:
        print('[Attn] Using PyTorch SDPA fallback')
os.environ['ATTN_BACKEND'] = attn_backend
os.environ['SPARSE_ATTN_BACKEND'] = attn_backend
os.environ['FLEX_GEMM_AUTOTUNE_CACHE_PATH'] = '/content/trellis2_autotune_cache.json'

import o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline, Trellis2TexturingPipeline
from trellis2.modules import image_feature_extractor
from trellis2.pipelines.rembg import BiRefNet as _BiRefNetMod
from trellis2.renderers import EnvMap
from trellis2.utils import render_utils

# Patch DinoV3FeatureExtractor & BiRefNet so gated HF repos automatically fall back to ungated mirrors
_orig_dinov3_init = image_feature_extractor.DinoV3FeatureExtractor.__init__
def _safe_dinov3_init(self, model_name: str, image_size: int = 512):
    try:
        _orig_dinov3_init(self, model_name=model_name, image_size=image_size)
    except Exception as e:
        fallback = 'camenduru/dinov3-vitl16-pretrain-lvd1689m'
        print(f"[DinoV3] {model_name} unavailable ({e}); falling back to ungated {fallback}")
        _orig_dinov3_init(self, model_name=fallback, image_size=image_size)
image_feature_extractor.DinoV3FeatureExtractor.__init__ = _safe_dinov3_init

_orig_birefnet_init = _BiRefNetMod.BiRefNet.__init__
def _safe_birefnet_init(self, model_name: str = "ZhengPeng7/BiRefNet"):
    try:
        _orig_birefnet_init(self, model_name=model_name)
    except Exception as e:
        fallback = "ZhengPeng7/BiRefNet"
        print(f"[BiRefNet] {model_name} unavailable ({e}); falling back to ungated {fallback}")
        _orig_birefnet_init(self, model_name=fallback)
_BiRefNetMod.BiRefNet.__init__ = _safe_birefnet_init

MODEL_REPO_ID = "microsoft/TRELLIS.2-4B"
OUTPUT_DIR = "/content/trellis2_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pipeline_i23d = None
pipeline_tex = None
envmaps = None
_active_low_vram = None
_active_tex_low_vram = None

def cleanup_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def load_envmaps(device='cuda'):
    global envmaps
    if envmaps is not None:
        return envmaps
    envmaps = {}
    hdri_base = os.path.join(REPO_DIR, 'assets', 'hdri')
    for name in ('forest', 'sunset', 'courtyard'):
        p = os.path.join(hdri_base, f'{name}.exr')
        if os.path.isfile(p):
            img_bgr = cv2.imread(p, cv2.IMREAD_UNCHANGED)
            if img_bgr is not None:
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                envmaps[name] = EnvMap(torch.tensor(img_rgb, dtype=torch.float32, device=device))
    print(f"[EnvMap] Loaded {len(envmaps)} HDRI environment map(s): {list(envmaps.keys())}")
    return envmaps

def init_trellis2(low_vram=None):
    """Load Trellis2ImageTo3DPipeline with automatic or explicit low_vram management."""
    global pipeline_i23d, pipeline_tex, _active_low_vram
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if low_vram is None:
        low_vram = vram_gb < 28.0
    low_vram = bool(low_vram)

    if pipeline_i23d is not None and _active_low_vram == low_vram:
        return pipeline_i23d

    if pipeline_tex is not None and low_vram:
        print("[VRAM] Unloading texturing pipeline to free memory for Image-to-3D...")
        pipeline_tex = None
        cleanup_vram()

    print("=" * 64)
    print(f"[Init] Loading {MODEL_REPO_ID} | Low-VRAM Offload: {low_vram} ({vram_gb:.1f} GB GPU)")
    print("=" * 64)
    t0 = time.time()
    if pipeline_i23d is None:
        pipeline_i23d = Trellis2ImageTo3DPipeline.from_pretrained(MODEL_REPO_ID)

    pipeline_i23d.low_vram = low_vram
    if low_vram:
        pipeline_i23d.cpu()
        pipeline_i23d.image_cond_model.cpu()
        if pipeline_i23d.rembg_model is not None:
            pipeline_i23d.rembg_model.cpu()
        pipeline_i23d._device = torch.device('cuda')
    else:
        pipeline_i23d.cuda()

    _active_low_vram = low_vram
    print(f"[Init] Trellis2ImageTo3DPipeline ready in {time.time() - t0:.1f}s")
    return pipeline_i23d

def init_texturing_pipeline(low_vram=None):
    """Load Trellis2TexturingPipeline for shape-conditioned PBR texture generation."""
    global pipeline_tex, pipeline_i23d, _active_tex_low_vram
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if low_vram is None:
        low_vram = vram_gb < 28.0
    low_vram = bool(low_vram)

    if pipeline_tex is not None and _active_tex_low_vram == low_vram:
        return pipeline_tex

    if pipeline_i23d is not None and low_vram:
        print("[VRAM] Unloading Image-to-3D pipeline to free memory for Texturing...")
        pipeline_i23d = None
        cleanup_vram()

    print(f"[Init] Loading Trellis2TexturingPipeline ({MODEL_REPO_ID}) | Low-VRAM: {low_vram}")
    t0 = time.time()
    if pipeline_tex is None:
        pipeline_tex = Trellis2TexturingPipeline.from_pretrained(
            MODEL_REPO_ID, config_file="texturing_pipeline.json"
        )
    pipeline_tex.low_vram = low_vram
    if low_vram:
        pipeline_tex.cpu()
        pipeline_tex.image_cond_model.cpu()
        if pipeline_tex.rembg_model is not None:
            pipeline_tex.rembg_model.cpu()
        pipeline_tex._device = torch.device('cuda')
    else:
        pipeline_tex.cuda()
    _active_tex_low_vram = low_vram
    print(f"[Init] Trellis2TexturingPipeline ready in {time.time() - t0:.1f}s")
    return pipeline_tex

def run_trellis2_image_to_3d(
    image_input,
    resolution_preset="1024_cascade",
    seed=42,
    low_vram=None,
    preprocess_image=True,
    decimation_target=500000,
    texture_size=2048,
    remesh=True,
    remesh_band=1,
    remesh_project=0,
    ss_steps=12,
    ss_guidance=7.5,
    ss_rescale=0.7,
    ss_rescale_t=5.0,
    shape_steps=12,
    shape_guidance=7.5,
    shape_rescale=0.5,
    shape_rescale_t=3.0,
    tex_steps=12,
    tex_guidance=1.0,
    tex_rescale=0.0,
    tex_rescale_t=3.0,
    max_num_tokens=49152,
    render_turntable=True,
    hdri_env="forest",
    output_glb_path=None,
):
    """Run Microsoft TRELLIS.2 4B Image-to-PBR-3D pipeline and export GLB + optional PBR turntable video."""
    pipe = init_trellis2(low_vram=low_vram)
    t_start = time.time()

    if isinstance(image_input, (str, os.PathLike, pathlib.Path)):
        stem = pathlib.Path(image_input).stem
        pil_in = Image.open(image_input)
    elif isinstance(image_input, np.ndarray):
        stem = "input"
        pil_in = Image.fromarray(image_input)
    elif isinstance(image_input, Image.Image):
        stem = "input"
        pil_in = image_input
    else:
        raise ValueError(f"Unsupported image_input type: {type(image_input)}")

    # Map resolution preset to valid TRELLIS.2 pipeline_type
    preset_map = {
        "512": "512",
        "1024": "1024_cascade",
        "1024_cascade": "1024_cascade",
        "1024_direct": "1024",
        "1536": "1536_cascade",
        "1536_cascade": "1536_cascade",
    }
    pipe_type = preset_map.get(str(resolution_preset), "1024_cascade")

    # Preprocess image (alpha matting via BiRefNet + bounding-box crop)
    if preprocess_image:
        fg_image = pipe.preprocess_image(pil_in)
    else:
        fg_image = pil_in.convert("RGB")

    ss_params = {
        "steps": int(ss_steps),
        "guidance_strength": float(ss_guidance),
        "guidance_rescale": float(ss_rescale),
        "rescale_t": float(ss_rescale_t),
    }
    shape_params = {
        "steps": int(shape_steps),
        "guidance_strength": float(shape_guidance),
        "guidance_rescale": float(shape_rescale),
        "rescale_t": float(shape_rescale_t),
    }
    tex_params = {
        "steps": int(tex_steps),
        "guidance_strength": float(tex_guidance),
        "guidance_rescale": float(tex_rescale),
        "rescale_t": float(tex_rescale_t),
    }

    print(f"[TRELLIS.2] Generating 3D asset | preset={pipe_type} | seed={seed} | low_vram={pipe.low_vram}")
    outputs, (shape_slat, tex_slat, res) = pipe.run(
        fg_image,
        num_samples=1,
        seed=int(seed),
        sparse_structure_sampler_params=ss_params,
        shape_slat_sampler_params=shape_params,
        tex_slat_sampler_params=tex_params,
        preprocess_image=False,
        return_latent=True,
        pipeline_type=pipe_type,
        max_num_tokens=int(max_num_tokens),
    )
    mesh = outputs[0]
    cleanup_vram()

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    if output_glb_path is None:
        output_glb_path = os.path.join(OUTPUT_DIR, f"{stem}_{pipe_type}_{ts}.glb")
    os.makedirs(os.path.dirname(output_glb_path) or OUTPUT_DIR, exist_ok=True)

    # Optional PBR turntable video rendering before GLB extraction
    video_path = None
    snapshot_path = None
    if render_turntable:
        try:
            print(f"[Render] Rendering PBR turntable video (env={hdri_env})...")
            mesh.simplify(16777216)  # nvdiffrast triangle limit
            em_dict = load_envmaps(device='cuda')
            selected_env = em_dict.get(hdri_env) or next(iter(em_dict.values()), None)
            if selected_env is not None:
                raw_vid = render_utils.render_video(mesh, envmap=selected_env, resolution=512, num_frames=60)
                vis_frames = render_utils.make_pbr_vis_frames(raw_vid)
                video_path = output_glb_path.replace(".glb", "_pbr_turntable.mp4")
                imageio.mimsave(video_path, vis_frames, fps=15)
                snapshot_path = output_glb_path.replace(".glb", "_preview.png")
                Image.fromarray(vis_frames[len(vis_frames) // 4]).save(snapshot_path)
                print(f"[Render] Saved PBR video -> {video_path}")
        except Exception as e:
            print(f"[Render] Turntable video skipped: {e}")
        finally:
            cleanup_vram()

    # Convert O-Voxel representation + mesh to UV-baked PBR GLB
    print(f"[O-Voxel] Baking PBR GLB (decimation_target={decimation_target}, texture_size={texture_size}, remesh={remesh})...")
    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices,
        faces=mesh.faces,
        attr_volume=mesh.attrs,
        coords=mesh.coords,
        attr_layout=pipe.pbr_attr_layout,
        grid_size=res,
        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target=int(decimation_target),
        texture_size=int(texture_size),
        remesh=bool(remesh),
        remesh_band=int(remesh_band),
        remesh_project=int(remesh_project),
        use_tqdm=True,
    )
    glb.export(output_glb_path, extension_webp=True)
    del mesh, outputs, shape_slat, tex_slat
    cleanup_vram()

    elapsed = time.time() - t_start
    sz_mb = os.path.getsize(output_glb_path) / (1024 * 1024) if os.path.exists(output_glb_path) else 0.0
    print(f"[TRELLIS.2] Saved GLB -> {output_glb_path} ({sz_mb:.2f} MB) in {elapsed:.1f}s")
    return {
        "glb_path": output_glb_path,
        "video_path": video_path,
        "snapshot_path": snapshot_path,
        "preprocessed_image": fg_image,
        "grid_size": int(res),
        "elapsed_s": round(elapsed, 1),
        "size_mb": round(sz_mb, 2),
    }

def run_trellis2_texturing(
    mesh_path,
    image_input,
    resolution=1024,
    seed=42,
    low_vram=None,
    texture_size=2048,
    output_glb_path=None,
):
    """Run shape-conditioned PBR texture synthesis on an existing 3D mesh using Trellis2TexturingPipeline."""
    pipe_t = init_texturing_pipeline(low_vram=low_vram)
    t_start = time.time()

    mesh = trimesh.load(str(mesh_path), force='mesh')
    if isinstance(image_input, (str, os.PathLike, pathlib.Path)):
        pil_in = Image.open(image_input)
    elif isinstance(image_input, np.ndarray):
        pil_in = Image.fromarray(image_input)
    else:
        pil_in = image_input

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = pathlib.Path(mesh_path).stem
    if output_glb_path is None:
        output_glb_path = os.path.join(OUTPUT_DIR, f"{stem}_pbr_textured_{ts}.glb")

    print(f"[TRELLIS.2 Texturing] Synthesizing PBR textures for {stem} at resolution={resolution}...")
    textured_glb = pipe_t.run(
        mesh,
        pil_in,
        seed=int(seed),
        resolution=int(resolution),
        texture_size=int(texture_size),
    )
    textured_glb.export(output_glb_path, extension_webp=True)
    cleanup_vram()
    elapsed = time.time() - t_start
    print(f"[TRELLIS.2 Texturing] Saved -> {output_glb_path} ({elapsed:.1f}s)")
    return output_glb_path

print("[Step 3] TRELLIS.2 core functions defined and ready.")

In [ ]:
# @title STEP 4 — Launch Gradio UI (Image-to-PBR-3D & Shape Texturing)
# @markdown Interactive Gradio interface with resolution presets (`512`, `1024_cascade`, `1536_cascade`), Low-VRAM CPU offload toggle, PBR texture baking controls (`texture_size`, `decimation_target`, `remesh`), interactive 3D `.glb` viewer, and PBR turntable preview video.

import os, sys, glob, random, traceback
import torch
import gradio as gr
from IPython.display import clear_output

REPO_DIR = '/content/TRELLIS.2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
gpu_name = torch.cuda.get_device_name(0)
default_low_vram = vram_gb < 28.0
default_preset = "1024_cascade" if vram_gb >= 15.0 else "512"

example_dir = os.path.join(REPO_DIR, "assets", "example_image")
EXAMPLE_IMAGES = sorted(
    glob.glob(os.path.join(example_dir, "*.png"))
    + glob.glob(os.path.join(example_dir, "*.webp"))
    + glob.glob(os.path.join(example_dir, "*.jpg"))
)[:12] if os.path.isdir(example_dir) else []

def ui_generate_3d(
    image_in,
    resolution_preset,
    low_vram,
    seed,
    randomize_seed,
    preprocess_bg,
    texture_size,
    decimation_target,
    remesh,
    remesh_band,
    render_turntable,
    hdri_env,
    ss_steps,
    ss_guidance,
    ss_rescale,
    ss_rescale_t,
    shape_steps,
    shape_guidance,
    shape_rescale,
    shape_rescale_t,
    tex_steps,
    tex_guidance,
    tex_rescale,
    tex_rescale_t,
    progress=gr.Progress(track_tqdm=True),
):
    if image_in is None:
        raise gr.Error("Please upload an input image or click one of the example images below.")
    try:
        if randomize_seed:
            seed = random.randint(0, 2**31 - 1)
        progress(0.05, desc=f"Initializing TRELLIS.2-4B ({resolution_preset})...")
        res = run_trellis2_image_to_3d(
            image_input=image_in,
            resolution_preset=resolution_preset,
            seed=int(seed),
            low_vram=bool(low_vram),
            preprocess_image=bool(preprocess_bg),
            decimation_target=int(decimation_target),
            texture_size=int(texture_size),
            remesh=bool(remesh),
            remesh_band=int(remesh_band),
            ss_steps=int(ss_steps),
            ss_guidance=float(ss_guidance),
            ss_rescale=float(ss_rescale),
            ss_rescale_t=float(ss_rescale_t),
            shape_steps=int(shape_steps),
            shape_guidance=float(shape_guidance),
            shape_rescale=float(shape_rescale),
            shape_rescale_t=float(shape_rescale_t),
            tex_steps=int(tex_steps),
            tex_guidance=float(tex_guidance),
            tex_rescale=float(tex_rescale),
            tex_rescale_t=float(tex_rescale_t),
            render_turntable=bool(render_turntable),
            hdri_env=str(hdri_env),
        )
        stats = {
            "status": "ok",
            "resolution_preset": resolution_preset,
            "final_grid_size": res["grid_size"],
            "seed": int(seed),
            "low_vram": bool(low_vram),
            "texture_size": int(texture_size),
            "decimation_target": int(decimation_target),
            "remesh": bool(remesh),
            "elapsed_seconds": res["elapsed_s"],
            "glb_size_mb": res["size_mb"],
            "glb_path": res["glb_path"],
        }
        return (
            res["glb_path"],
            res["video_path"],
            res["preprocessed_image"],
            res["glb_path"],
            stats,
            int(seed),
        )
    except Exception as e:
        traceback.print_exc()
        return None, None, None, None, {"status": "error", "message": str(e)}, int(seed)

def ui_texture_mesh(mesh_file, ref_image, tex_res, tex_size, seed, low_vram, progress=gr.Progress(track_tqdm=True)):
    if mesh_file is None or ref_image is None:
        raise gr.Error("Please provide both an input 3D mesh (.glb/.obj/.ply) and a reference image.")
    try:
        progress(0.1, desc="Loading Trellis2TexturingPipeline...")
        mesh_path = mesh_file.name if hasattr(mesh_file, "name") else str(mesh_file)
        glb_out = run_trellis2_texturing(
            mesh_path=mesh_path,
            image_input=ref_image,
            resolution=int(tex_res),
            seed=int(seed),
            low_vram=bool(low_vram),
            texture_size=int(tex_size),
        )
        return glb_out, glb_out, {"status": "ok", "output_glb": glb_out}
    except Exception as e:
        traceback.print_exc()
        return None, None, {"status": "error", "message": str(e)}

with gr.Blocks(title="Microsoft TRELLIS.2 (4B) — Image-to-PBR-3D", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        f"# 🧊 Microsoft TRELLIS.2 (4B) — Image-to-PBR-3D (O-Voxel)\n"
        f"**GPU:** `{gpu_name}` ({vram_gb:.1f} GB VRAM) &nbsp;|&nbsp; **License:** MIT &nbsp;|&nbsp; "
        f"**Default Mode:** `{'Low-VRAM CPU Offload' if default_low_vram else 'Full GPU Standard'}`"
    )

    with gr.Tabs():
        with gr.TabItem("1. Image to PBR 3D (Trellis2ImageTo3DPipeline)"):
            with gr.Row():
                with gr.Column(scale=5):
                    img_in = gr.Image(
                        label="Input Image (RGBA or RGB)",
                        type="pil",
                        image_mode="RGBA",
                        height=300,
                        info="Upload an object or character image. Transparent PNGs use their alpha channel directly; RGB images are matted with BiRefNet.",
                    )
                    with gr.Row():
                        resolution_preset = gr.Dropdown(
                            choices=["512", "1024_cascade", "1536_cascade"],
                            value=default_preset,
                            label="Resolution Preset",
                            info="512 = fastest (~15s); 1024_cascade = balanced high-detail default; 1536_cascade = ultra-sharp O-Voxel cascade.",
                        )
                        low_vram = gr.Checkbox(
                            value=default_low_vram,
                            label="Low-VRAM / CPU Offload",
                            info="Offloads inactive DiT stages to CPU RAM between steps. Recommended on T4 (15 GB) and L4 (24 GB).",
                        )
                    with gr.Row():
                        preprocess_bg = gr.Checkbox(
                            value=True,
                            label="Auto Remove Background & Center",
                            info="Run BiRefNet background removal and tight bounding-box centering before conditioning.",
                        )
                        render_turntable = gr.Checkbox(
                            value=True,
                            label="Render PBR Turntable Video",
                            info="Render a 60-frame PBR + Normal turntable preview video alongside the GLB mesh.",
                        )

                    with gr.Accordion("Texture Baking & O-Voxel Mesh Post-Processing", open=True):
                        with gr.Row():
                            texture_size = gr.Dropdown(
                                choices=[1024, 2048, 4096],
                                value=2048,
                                label="PBR Texture Size (px)",
                                info="Resolution of baked BaseColor, Metallic, Roughness, and Alpha UV maps (1024–4096).",
                            )
                            decimation_target = gr.Slider(
                                minimum=50000,
                                maximum=1000000,
                                value=500000,
                                step=25000,
                                label="Decimation Target (Faces)",
                                info="Target triangle count for CuMesh simplification before UV unwrapping.",
                            )
                        with gr.Row():
                            remesh = gr.Checkbox(
                                value=True,
                                label="Topological Remeshing",
                                info="Clean and regularize O-Voxel dual-grid topology before UV baking.",
                            )
                            remesh_band = gr.Slider(
                                minimum=1,
                                maximum=4,
                                value=1,
                                step=1,
                                label="Remesh Band Width",
                                info="Narrow-band width for dual-contouring remesh projection (1 = default).",
                            )
                            hdri_env = gr.Dropdown(
                                choices=["forest", "sunset", "courtyard"],
                                value="forest",
                                label="HDRI Lighting Environment",
                                info="Environment map used for the PBR turntable preview video.",
                            )

                    with gr.Accordion("Seed & 3-Stage Flow-Matching Sampler Settings", open=False):
                        with gr.Row():
                            seed = gr.Number(
                                value=42,
                                precision=0,
                                label="Random Seed",
                                info="Deterministic seed for all 3 flow-matching stages.",
                            )
                            randomize_seed = gr.Checkbox(
                                value=False,
                                label="Randomize Seed on Generate",
                                info="Pick a fresh random seed on every run.",
                            )
                        gr.Markdown("**Stage 1: Sparse Structure DiT**")
                        with gr.Row():
                            ss_steps = gr.Slider(4, 30, value=12, step=1, label="SS Steps", info="Diffusion steps for 3D sparse occupancy grid.")
                            ss_guidance = gr.Slider(1.0, 12.0, value=7.5, step=0.5, label="SS CFG", info="Classifier-free guidance strength for Stage 1.")
                            ss_rescale = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="SS Rescale", info="Guidance rescale factor for Stage 1 (0.7 default).")
                            ss_rescale_t = gr.Slider(1.0, 8.0, value=5.0, step=0.5, label="SS Rescale T", info="Timestep rescale schedule for Stage 1.")
                        gr.Markdown("**Stage 2: Shape SLat DiT**")
                        with gr.Row():
                            shape_steps = gr.Slider(4, 30, value=12, step=1, label="Shape Steps", info="Diffusion steps for O-Voxel shape structured latent.")
                            shape_guidance = gr.Slider(1.0, 12.0, value=7.5, step=0.5, label="Shape CFG", info="Classifier-free guidance strength for Stage 2.")
                            shape_rescale = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Shape Rescale", info="Guidance rescale factor for Stage 2 (0.5 default).")
                            shape_rescale_t = gr.Slider(1.0, 8.0, value=3.0, step=0.5, label="Shape Rescale T", info="Timestep rescale schedule for Stage 2.")
                        gr.Markdown("**Stage 3: Material / PBR Texture SLat DiT**")
                        with gr.Row():
                            tex_steps = gr.Slider(4, 30, value=12, step=1, label="Material Steps", info="Diffusion steps for PBR material structured latent.")
                            tex_guidance = gr.Slider(1.0, 10.0, value=1.0, step=0.5, label="Material CFG", info="Classifier-free guidance for Stage 3 (1.0 default).")
                            tex_rescale = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Material Rescale", info="Guidance rescale for Stage 3 (0.0 default).")
                            tex_rescale_t = gr.Slider(1.0, 8.0, value=3.0, step=0.5, label="Material Rescale T", info="Timestep rescale schedule for Stage 3.")

                    btn_gen = gr.Button("🧊 Generate PBR 3D Asset (.glb)", variant="primary", size="lg")

                with gr.Column(scale=6):
                    out_model3d = gr.Model3D(
                        label="Interactive 3D PBR Mesh (.glb)",
                        height=420,
                        clear_color=(0.18, 0.18, 0.22, 1.0),
                    )
                    with gr.Row():
                        out_video = gr.Video(
                            label="PBR + Normal Turntable Preview (.mp4)",
                            height=240,
                        )
                        out_fg = gr.Image(
                            label="Preprocessed Foreground",
                            type="pil",
                            height=240,
                        )
                    out_file = gr.File(
                        label="Download Baked PBR .glb",
                        info="Download the WebP-compressed .glb with BaseColor, Metallic, Roughness, and Opacity maps.",
                    )
                    out_stats = gr.JSON(label="Generation Metadata")

            if EXAMPLE_IMAGES:
                gr.Examples(
                    examples=[[p] for p in EXAMPLE_IMAGES],
                    inputs=[img_in],
                    label="Built-in TRELLIS.2 Example Images",
                    examples_per_page=6,
                )

            btn_gen.click(
                fn=ui_generate_3d,
                inputs=[
                    img_in, resolution_preset, low_vram, seed, randomize_seed,
                    preprocess_bg, texture_size, decimation_target, remesh, remesh_band,
                    render_turntable, hdri_env,
                    ss_steps, ss_guidance, ss_rescale, ss_rescale_t,
                    shape_steps, shape_guidance, shape_rescale, shape_rescale_t,
                    tex_steps, tex_guidance, tex_rescale, tex_rescale_t,
                ],
                outputs=[out_model3d, out_video, out_fg, out_file, out_stats, seed],
            )

        with gr.TabItem("2. Shape-Conditioned PBR Texturing (Trellis2TexturingPipeline)"):
            gr.Markdown(
                "Upload an untextured `.glb / .obj / .ply` mesh and a reference image to synthesize "
                "3D-consistent PBR materials using `Trellis2TexturingPipeline` (`texturing_pipeline.json`)."
            )
            with gr.Row():
                with gr.Column(scale=5):
                    mesh_in_t = gr.File(
                        label="Input 3D Mesh (.glb, .obj, .ply)",
                        file_types=[".glb", ".obj", ".ply", ".stl"],
                        info="Untextured or white mesh to convert into O-Voxel and paint with PBR materials.",
                    )
                    img_in_t = gr.Image(
                        label="Reference Style / Concept Image",
                        type="pil",
                        image_mode="RGBA",
                        height=260,
                        info="Reference image guiding the PBR appearance of the input mesh.",
                    )
                    with gr.Row():
                        tex_res_t = gr.Radio(
                            choices=[512, 1024],
                            value=1024,
                            label="O-Voxel Encoding Resolution",
                            info="Voxel grid resolution for encoding the input mesh geometry.",
                        )
                        tex_size_t = gr.Dropdown(
                            choices=[1024, 2048, 4096],
                            value=2048,
                            label="Baked Texture Size (px)",
                            info="UV atlas resolution for the exported PBR GLB.",
                        )
                    with gr.Row():
                        seed_t = gr.Number(value=42, precision=0, label="Seed", info="Random seed for texture flow matching.")
                        low_vram_t = gr.Checkbox(value=default_low_vram, label="Low-VRAM Mode", info="Offload submodules to CPU between steps.")
                    btn_tex = gr.Button("🎨 Bake PBR Textures onto Mesh", variant="primary")
                with gr.Column(scale=6):
                    out_tex_model3d = gr.Model3D(label="Textured PBR Mesh (.glb)", height=420)
                    out_tex_file = gr.File(label="Download Textured .glb", info="Download the newly textured PBR GLB asset.")
                    out_tex_stats = gr.JSON(label="Texturing Status")

            btn_tex.click(
                fn=ui_texture_mesh,
                inputs=[mesh_in_t, img_in_t, tex_res_t, tex_size_t, seed_t, low_vram_t],
                outputs=[out_tex_model3d, out_tex_file, out_tex_stats],
            )

    demo.load(
        lambda: gr.Info("Welcome to Microsoft TRELLIS.2 (4B)! Upload an image or pick an example, choose a resolution preset, and click Generate."),
        inputs=None,
        outputs=None,
    )

clear_output()
demo.queue(max_size=4, concurrency_limit=1).launch(share=True, debug=False, show_error=True)

In [ ]:
# @title STEP 5 — Keep-Alive (Prevent Colab Timeout)
# @markdown Pings the Colab frontend every 30 minutes so long batch runs or interactive Gradio sessions do not idle out.

import IPython
from IPython.display import display

KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("TRELLIS.2 keep-alive:", new Date().toISOString());
        if (window.google && google.colab && google.colab.kernel) {
            google.colab.kernel.proxyPort(0).then(() => {}).catch(() => {});
        }
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print("[KeepAlive] Active — pinging Colab runtime every 30 minutes.")

In [ ]:
# @title STEP 6 — Quick Test (Single-Image to PBR `.glb` + Turntable Video)
# @markdown Generates one PBR `.glb` mesh and turntable `.mp4` directly in the notebook without launching Gradio. Leave `IMAGE_PATH` blank to auto-select a built-in example from `/content/TRELLIS.2/assets/example_image/`.

import os, glob, pathlib
from IPython.display import FileLink, display, Image as IPImage, Video as IPVideo

IMAGE_PATH = ""  # @param {type:"string"}
RESOLUTION_PRESET = "1024_cascade"  # @param ["512", "1024_cascade", "1536_cascade"]
LOW_VRAM = True  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}
TEXTURE_SIZE = 2048  # @param [1024, 2048, 4096] {type:"raw"}
DECIMATION_TARGET = 500000  # @param {type:"slider", min:50000, max:1000000, step:50000}
REMESH = True  # @param {type:"boolean"}
RENDER_TURNTABLE = True  # @param {type:"boolean"}
HDRI_ENV = "forest"  # @param ["forest", "sunset", "courtyard"]

if not IMAGE_PATH.strip():
    candidates = [
        "/content/TRELLIS.2/assets/example_image/T.png",
        "/content/upload.png",
        "/content/input.png",
    ] + sorted(glob.glob("/content/TRELLIS.2/assets/example_image/*.*"))
    IMAGE_PATH = next((p for p in candidates if os.path.isfile(p)), "")

if not IMAGE_PATH or not os.path.isfile(IMAGE_PATH):
    raise SystemExit("[Step 6] No input image found. Upload an image to /content/ or check /content/TRELLIS.2/assets/example_image/.")

print(f"[Step 6] Running TRELLIS.2-4B Quick Test on: {IMAGE_PATH}")
result = run_trellis2_image_to_3d(
    image_input=IMAGE_PATH,
    resolution_preset=RESOLUTION_PRESET,
    seed=SEED,
    low_vram=LOW_VRAM,
    decimation_target=DECIMATION_TARGET,
    texture_size=TEXTURE_SIZE,
    remesh=REMESH,
    render_turntable=RENDER_TURNTABLE,
    hdri_env=HDRI_ENV,
)

if result.get("snapshot_path") and os.path.isfile(result["snapshot_path"]):
    display(IPImage(filename=result["snapshot_path"]))

if result.get("video_path") and os.path.isfile(result["video_path"]):
    print(f"[Step 6] Turntable Video: {result['video_path']}")
    display(FileLink(result["video_path"]))

glb_path = result["glb_path"]
print(f"[Step 6] Generated PBR GLB ({result['size_mb']} MB in {result['elapsed_s']}s): {glb_path}")
display(FileLink(glb_path))

In [ ]:
# @title STEP 7 — Batch Process Folder of Images to PBR `.glb` Assets
# @markdown Batch-converts all images in `BATCH_INPUT_DIR` into PBR-textured `.glb` meshes (and optional `.mp4` turntable previews) with per-item error handling and `tqdm` progress tracking.

import os, glob, shutil, pathlib, time
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_INPUT_DIR = "/content/images_in"  # @param {type:"string"}
BATCH_OUTPUT_DIR = "/content/drive/MyDrive/AEI_3D_Out/TRELLIS2_Batch"  # @param {type:"string"}
RESOLUTION_PRESET = "1024_cascade"  # @param ["512", "1024_cascade", "1536_cascade"]
LOW_VRAM = True  # @param {type:"boolean"}
TEXTURE_SIZE = 2048  # @param [1024, 2048, 4096] {type:"raw"}
DECIMATION_TARGET = 500000  # @param {type:"slider", min:50000, max:1000000, step:50000}
REMESH = True  # @param {type:"boolean"}
RENDER_TURNTABLE = False  # @param {type:"boolean"}
MAX_ITEMS = 0  # @param {type:"integer"}

if not os.path.exists("/content/drive/MyDrive") and BATCH_OUTPUT_DIR.startswith("/content/drive"):
    BATCH_OUTPUT_DIR = "/content/trellis2_batch_output"

os.makedirs(BATCH_INPUT_DIR, exist_ok=True)
os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

exts = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.PNG", "*.JPG", "*.JPEG", "*.WEBP")
image_files = []
for pat in exts:
    image_files.extend(glob.glob(os.path.join(BATCH_INPUT_DIR, pat)))
image_files = sorted(set(image_files))

if not image_files:
    example_fallback = sorted(glob.glob("/content/TRELLIS.2/assets/example_image/*.*"))[:3]
    print(f"[Batch] {BATCH_INPUT_DIR} is empty; using {len(example_fallback)} sample image(s) from TRELLIS.2 assets.")
    image_files = example_fallback

if MAX_ITEMS > 0:
    image_files = image_files[:MAX_ITEMS]

print(f"[Batch] Processing {len(image_files)} image(s) -> {BATCH_OUTPUT_DIR}")
summary = []

for idx, img_path in enumerate(tqdm(image_files, desc="TRELLIS.2 Batch"), 1):
    stem = pathlib.Path(img_path).stem
    out_glb = os.path.join(BATCH_OUTPUT_DIR, f"{idx:03d}_{stem}.glb")
    if os.path.isfile(out_glb) and os.path.getsize(out_glb) > 1024:
        print(f"  [{idx}/{len(image_files)}] SKIP (already exists): {out_glb}")
        summary.append((stem, "skipped", out_glb))
        continue
    try:
        res = run_trellis2_image_to_3d(
            image_input=img_path,
            resolution_preset=RESOLUTION_PRESET,
            seed=42 + idx,
            low_vram=LOW_VRAM,
            decimation_target=DECIMATION_TARGET,
            texture_size=TEXTURE_SIZE,
            remesh=REMESH,
            render_turntable=RENDER_TURNTABLE,
            output_glb_path=out_glb,
        )
        summary.append((stem, "ok", res["glb_path"]))
    except Exception as e:
        print(f"  [{idx}/{len(image_files)}] ERROR on {img_path}: {e}")
        summary.append((stem, "error", str(e)))
    finally:
        cleanup_vram()

ok_count = sum(1 for _, s, _ in summary if s == "ok")
skip_count = sum(1 for _, s, _ in summary if s == "skipped")
err_count = sum(1 for _, s, _ in summary if s == "error")
print(f"\n[Batch Done] {ok_count} succeeded, {skip_count} skipped, {err_count} failed. Output folder: {BATCH_OUTPUT_DIR}")
for stem, status, path in summary:
    if status == "ok" and os.path.isfile(path):
        display(FileLink(path))